# Assignment 3 - Hierarchical clustering
Compare WT_AL and WT_RF using Ward clustering with Euclidean distance. Run the cells in order.

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from scipy.cluster.hierarchy import linkage, cut_tree, dendrogram
from sklearn.preprocessing import StandardScaler
from pathlib import Path

results = Path('results/assignment3/hierarchical')
results.mkdir(parents=True, exist_ok=True)

gene_counts = [10, 100, 1000, 5000, 10000]
k_values = [2, 3, 4, 5]
plot_k = 3  # Example cut for the dendrogram.

## Load data

In [ ]:
df = pd.read_csv('Data/SRP064298/gene_data.tsv', sep='\t')
metadata = pd.read_csv('Data/SRP064298/metadata_SRP064298.tsv', sep='\t')
metadata = metadata.set_index('refinebio_accession_code')

# Match the metadata to the expression columns.
metadata = metadata.loc[df.columns[1:]]
# Keep only wild-type AL/RF samples, excluding KO samples.
wt_samples = metadata['refinebio_title'].str.startswith(('WT_AL_', 'WT_RF_'))
metadata = metadata.loc[wt_samples].copy()
metadata.index.name = 'sample_id'

# Extract AL or RF from titles such as WT_AL_ZT00_A_RNA.
groups = metadata['refinebio_title'].str.split('_').str[1].rename('group')
# Transpose to samples x genes, then keep the selected samples.
samples = df.drop(columns='Gene').T.loc[metadata.index]
samples.index.name = 'sample_id'

# Check sample alignment and missing or infinite expression values.
assert samples.index.equals(groups.index)
assert np.isfinite(samples.to_numpy()).all()

print('Samples:', len(samples))
print(groups.value_counts())
print('Distinct sample titles:', metadata['refinebio_title'].nunique())

## Select variable genes
Use the supplied normalized values. Select by variance before scaling. Keep row numbers so repeated gene names remain separate.

In [ ]:
gene_variance = samples.var(axis=0)
gene_order = gene_variance[gene_variance > 0].sort_values(
    ascending=False, kind='mergesort'
).index

gene_ranking = pd.DataFrame({
    'gene_symbol': df['Gene'],
    'input_row': df.index,
    'variance': gene_variance,
}).loc[gene_order]
gene_ranking.index.name = 'gene_row'

for n_genes in gene_counts:
    gene_ranking.iloc[:n_genes].to_csv(results / f'top_{n_genes}_genes.csv')

gene_ranking.head(10)

## Run hierarchical clustering
Build one tree per gene count, then cut it at each k.


In [ ]:
trees = {}
cluster_labels = pd.DataFrame(index=samples.index)
summaries = []

for n_genes in gene_counts:
    selected_genes = gene_order[:n_genes]
    scaled_data = StandardScaler().fit_transform(samples.loc[:, selected_genes])
    trees[n_genes] = linkage(scaled_data, method='ward', metric='euclidean')
    cuts = cut_tree(trees[n_genes], n_clusters=k_values)

    for i, k in enumerate(k_values):
        labels = cuts[:, i] + 1
        run_name = f'hierarchical_g{n_genes}_k{k}'
        cluster_labels[run_name] = labels
        sizes = pd.Series(labels).value_counts()

        summaries.append({
            'run': run_name,
            'n_genes': n_genes,
            'k': k,
            'n_runs': len(labels),
            'smallest_cluster': sizes.min(),
            'largest_cluster': sizes.max(),
        })

    print(f'Finished {n_genes} genes')

summary = pd.DataFrame(summaries)
cluster_labels.to_csv(results / 'cluster_labels.csv')
summary.to_csv(results / 'run_summary.csv', index=False)
summary

## Dendrogram - 5,000 genes

In [ ]:
sample_tree = trees[5000]
cut_height = (sample_tree[-plot_k, 2] + sample_tree[-(plot_k - 1), 2]) / 2

plt.figure(figsize=(12, 5))
dendrogram(sample_tree, no_labels=True, color_threshold=cut_height)
plt.axhline(cut_height, color='black', linestyle='--', label=f'k={plot_k}')
plt.xlabel('Sequencing runs')
plt.ylabel('Ward merge distance')
plt.title('Dendrogram - 5,000 genes')
plt.legend()
plt.tight_layout()
plt.savefig(results / 'sample_dendrogram_full.png', dpi=180)
plt.show()
plt.close()